# byte pair encoding


In [12]:
import tiktoken
print("version:", tiktoken.__version__)


version: 0.14.0


Once installed, we can instantiate the BPE tokenizer from tiktoken as follows:

In [13]:
tokenizer = tiktoken.get_encoding("gpt2")

In [14]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [15]:
string = tokenizer.decode(integers)
print(string)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [16]:
print("vocab size:", tokenizer.n_vocab)

vocab size: 50257


lets load our dataset and apply and check how many token does it have


In [17]:
from datasets import load_from_disk

dataset = load_from_disk(
    r"C:\LLM from Scratch\datasets\tiny_stories_dataset"
)

In [18]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

In [19]:
dataset_text = " ".join(dataset["train"]['text'][:20000]) 

In [20]:
encode_dataset = tokenizer.encode(dataset_text, allowed_special={"<|endoftext|>"})

In [21]:
print("total tokens in the dataset:", len(encode_dataset))

total tokens in the dataset: 4443976


In [26]:
context_size = 4
x = encode_dataset[:context_size]
y = encode_dataset[1:context_size + 1]
print(f"x: {x}")
print(f"y:       {y}")

x: [3198, 1110, 11, 257]
y:       [1110, 11, 257, 1310]


In [ ]:
import torch 
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, data, tokenizer, max_length,stride):
        self.input_ids = []
        for text in data:
            inputs = tokenizer(text, return_tensors="pt", padding="max_length", truncation=True)
            self.input_ids.append(inputs)